In [1]:
import os
from pathlib import Path
from textSummarizer.entity import DataIngestionConfig
from textSummarizer.utils.common import read_yaml
from dataclasses import dataclass
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size
from box import ConfigBox
import requests

os.chdir(r"E:\text_summarizer")
print(os.getcwd())


E:\text_summarizer


In [2]:
PROJECT_ROOT = Path(r"E:\text_summarizer")
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"

print(CONFIG_PATH.exists())  # should be True


True


In [3]:
PROJECT_ROOT = Path(r"E:\text_summarizer")
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"


In [4]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
# Manually defining paths if constants import fails
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH):
    
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        
        create_directories([self.config.artifacts_root])
        
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir = Path(config.root_dir),
            source_URL = str(config.source_URL),
            local_data_file = Path(config.local_data_file),
            unzip_dir = Path(config.unzip_dir)
        )
        
        return data_ingestion_config    

In [9]:
%pip install requests -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            logger.info("Downloading started...")
            response = requests.get(self.config.source_URL, stream=True)
            response.raise_for_status()  # 🔥 THIS LINE IS CRITICAL

            with open(self.config.local_data_file, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            logger.info(f"File downloaded successfully at: {self.config.local_data_file}")
        else:
            logger.info(
            f"File already exists of size: {get_size(self.config.local_data_file)}"
        )

            
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        if not zipfile.is_zipfile(self.config.local_data_file):
            raise ValueError("Downloaded file is not a valid ZIP file")

        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)


In [8]:
try:    
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    print("🎉 Success! Check your artifacts folder now.")
except Exception as e:
    raise e

[2026-02-14 15:11:25,601: INFO: textSummarizer: YAML file: config\config.yaml loaded successfully]
[2026-02-14 15:11:25,621: INFO: textSummarizer: YAML file: params.yaml loaded successfully]
[2026-02-14 15:11:25,626: INFO: textSummarizer: created directory at: artifacts]
[2026-02-14 15:11:25,626: INFO: textSummarizer: created directory at: artifacts/data_ingestion]
[2026-02-14 15:11:25,629: INFO: textSummarizer: Downloading started...]
[2026-02-14 15:11:29,517: INFO: textSummarizer: File downloaded successfully at: artifacts\data_ingestion\data.zip]
🎉 Success! Check your artifacts folder now.
